# 001: Foundations, Lossy Monitoring, and the Geometry of Blind Spots

**Local Green, Global Red** — Notebook 1 of 3

**Core Topic:** How distributed systems with directed interfaces and lossy evaluators behave under composition, why monitoring is contractive (Theorem 1), and the exact geometric condition for All-Green (Theorem 2).

**AuthorityEffect:** None.


In [1]:
import sys
from pathlib import Path
import numpy as np

# Dynamically locate and import sibling code directory
nb_dir = Path.cwd().resolve()
candidates = [
    nb_dir.parent / "code",
    nb_dir / "code",
] + [p / "code" for p in nb_dir.parents if (p / "code" / "code_core.py").is_file()]

for c in candidates:
    if c.is_dir() and (c / "code_core.py").is_file():
        if str(c) not in sys.path:
            sys.path.insert(0, str(c))
        break

from code_core import (
    NormType, Site, Interface, Assignment, Evaluator, EvaluatorFamily, System,
    compute_norm, compute_defect, compute_global_inconsistency
)
from code_blind_spots import (
    compute_nullspace_basis, compute_interface_blind_spot_basis, is_in_blind_spot
)
from code_theorems import check_theorem_1_passivity, check_theorem_2_characterization

print("[OK] Modules loaded successfully from code/")

[OK] Modules loaded successfully from code/


## 1. The Physical & Computational Model

Let $S$ be a finite set of **sites** and $E$ a finite set of directed **interfaces** between them:
- Each site $x$ carries a real normed vector space $F(x)$ (e.g. $\mathbb{R}^k$).
- Each interface $e = (a, b)$ from site $a$ to site $b$ carries a non-expansive linear translation map $r_e: F(b) \to F(a)$ satisfying $\|r_e v\| \le \|v\|$.
- An **assignment** $s$ assigns a state vector $s(x) \in F(x)$ to each site $x$.

The **defect** across interface $e = (a, b)$ is:
$$\Delta(s)_e := r_e s(b) - s(a) \in F(a)$$

The **global inconsistency** is the total defect mass:
$$I(s) := \sum_{e \in E} \|\Delta(s)_e\|_1$$
- $I(s) = 0$ is **Global Green** (complete consistency across all interfaces).
- $I(s) > 0$ is **Global Red** (inconsistency exists somewhere in the system).

Let's build a simple 2-site system connected by one interface to see this in action.

In [2]:
# Define two sites with 2D state spaces
site_a = Site(name="site_A", dimension=2, norm_type=NormType.L1)
site_b = Site(name="site_B", dimension=2, norm_type=NormType.L1)

# Interface from A to B with identity translation (non-expansive)
r_e = np.eye(2)
interface_ab = Interface(name="e_AB", site_a="site_A", site_b="site_B", translation_matrix=r_e, norm_type=NormType.L1)

# Create an assignment where B has [3, 4] and A has [1, 2]
assignment = Assignment(states={
    "site_A": np.array([1.0, 2.0]),
    "site_B": np.array([3.0, 4.0]),
})

# Defect: r_e * s(B) - s(A) = [3, 4] - [1, 2] = [2, 2]
defect = compute_defect(interface_ab, assignment)
inconsistency = compute_norm(defect, NormType.L1)

print(f"Site A state:         {assignment.states['site_A']}")
print(f"Site B state:         {assignment.states['site_B']}")
print(f"Interface defect:     {defect}")
print(f"Global Inconsistency: {inconsistency} (L1 norm = |2| + |2| = 4.0)")

Site A state:         [1. 2.]
Site B state:         [3. 4.]
Interface defect:     [2. 2.]
Global Inconsistency: 4.0 (L1 norm = |2| + |2| = 4.0)


## 2. Evaluators and Lossy Measurement Operators

An **evaluator** $\alpha$ is defined by:
1. A subset of interfaces $E^\alpha \subseteq E$ that it monitors.
2. For each watched interface $e$, a linear measurement operator $q_e^\alpha: F(a) \to V_e^\alpha$ satisfying $\|q_e^\alpha v\| \le \|v\|$ (measurements do not amplify).

The **evaluator score** is:
$$J^\alpha(s) := \sum_{e \in E^\alpha} \|q_e^\alpha(\Delta(s)_e)\|$$
- $J^\alpha(s) = 0$ is **Local Green** for evaluator $\alpha$.
- The family is **All-Green** when $J^\alpha(s) = 0$ for all evaluators $\alpha$.

In [3]:
# Evaluator Alpha only observes the first coordinate (deleting coordinate 1)
# q(x, y) = [x] (1x2 measurement matrix, ||q|| <= 1)
q_alpha = np.array([[1.0, 0.0]])

evaluator_alpha = Evaluator(
    name="evaluator_alpha",
    watched_interfaces=["e_AB"],
    measurement_matrices={"e_AB": q_alpha},
    value_norms={"e_AB": NormType.L1}
)

system = System()
system.add_site(site_a)
system.add_site(site_b)
system.add_interface(interface_ab)
system.add_evaluator(evaluator_alpha)

scores = system.evaluator_scores(assignment)
print(f"True defect:           {defect}")
print(f"Global Inconsistency:  {inconsistency}")
print(f"Evaluator Alpha score: {scores['evaluator_alpha']} (observed only first coordinate = |2| = 2.0)")

True defect:           [2. 2.]
Global Inconsistency:  4.0
Evaluator Alpha score: 2.0 (observed only first coordinate = |2| = 2.0)


## 3. Theorem 1: Monitoring is One-Way (Evaluators Cannot Create Inconsistency)

**Theorem 1 (Passivity / Contraction).** For every assignment $s$ and every evaluator $\alpha$:
$$J^\alpha(s) \le I(s)$$

*Intuition:* Because measurements do not amplify ($\|q_e^\alpha v\| \le \|v\|$) and evaluators only sum over a subset of interfaces ($E^\alpha \subseteq E$), monitoring can only lose signal; it can never manufacture inconsistency out of thin air.

Let's test this inequality over 100 randomly sampled assignments:

In [4]:
np.random.seed(42)
max_ratio = 0.0

for trial in range(100):
    rnd_assignment = Assignment(states={
        "site_A": np.random.uniform(-10, 10, size=2),
        "site_B": np.random.uniform(-10, 10, size=2),
    })
    
    true_I = system.global_inconsistency(rnd_assignment)
    eval_scores = system.evaluator_scores(rnd_assignment)
    
    for name, score in eval_scores.items():
        assert score <= true_I + 1e-9, f"Violation! J={score} > I={true_I}"
        if true_I > 1e-9:
            max_ratio = max(max_ratio, score / true_I)

print(f"[PASSED] Passivity inequality held in 100 sampled assignments.")
print(f"Maximum observed ratio J^alpha(s) / I(s): {max_ratio:.4f} <= 1.0")

# Run verification suite assertion
assert check_theorem_1_passivity(system, assignment)
print("[OK] Numerical passivity check passed.")

[PASSED] Passivity inequality held in 100 sampled assignments.
Maximum observed ratio J^alpha(s) / I(s): 0.9889 <= 1.0
[OK] Numerical passivity check passed.


## 4. Theorem 2: The Exact Mechanism of False Green (Blind Spots)

The converse of Theorem 1 would be: *Does Local Green ($J = 0$) imply Global Green ($I = 0$)?*

**Theorem 2 (All-Green Characterized).** The family is All-Green at assignment $s$ **if and only if** at every interface $e$, the defect lies in that interface's **blind spot**:
$$\Delta(s)_e \in K_e \quad \forall e \in E$$
where the blind spot $K_e$ is the intersection of the measurement kernels of all evaluators watching $e$:
$$K_e := \bigcap_{\alpha: e \in E^\alpha} \ker(q_e^\alpha)$$
*(If an interface is unwatched, its blind spot is the entire state space $F(a)$.)*

Let's compute the blind spot basis for interface `e_AB` and construct an assignment where the defect lies strictly within the blind spot.

In [5]:
# Compute blind spot basis for e_AB
basis = compute_interface_blind_spot_basis("e_AB", interface_ab, system.evaluator_family)
print(f"Blind spot basis for e_AB: {basis}")
print(f"Dimension of blind spot:   {basis.shape[1]}")

# Any vector along [0, 1] is killed by evaluator Alpha!
# Let's create an assignment where the defect is [0, 5]
blind_assignment = Assignment(states={
    "site_A": np.array([0.0, 0.0]),
    "site_B": np.array([0.0, 5.0]), # defect = [0, 5]
})

blind_defect = compute_defect(interface_ab, blind_assignment)
blind_inconsistency = system.global_inconsistency(blind_assignment)
blind_eval_scores = system.evaluator_scores(blind_assignment)

print("\n--- Testing Blind Spot Assignment ---")
print(f"True Defect:               {blind_defect}")
print(f"Is defect in blind spot?   {is_in_blind_spot(blind_defect, basis)}")
print(f"True Global Inconsistency: {blind_inconsistency} (Global Red!)")
print(f"Evaluator Alpha score:     {blind_eval_scores['evaluator_alpha']} (Local Green!)")

# Numerical Theorem 2 diagnostic on this assignment
assert check_theorem_2_characterization(system, blind_assignment)
print("\n[OK] Theorem 2 holds: Evaluator reports 0.0 despite massive true inconsistency!")

Blind spot basis for e_AB: [[0.]
 [1.]]
Dimension of blind spot:   1

--- Testing Blind Spot Assignment ---
True Defect:               [0. 5.]
Is defect in blind spot?   True
True Global Inconsistency: 5.0 (Global Red!)
Evaluator Alpha score:     0.0 (Local Green!)

[OK] Theorem 2 holds: Evaluator reports 0.0 despite massive true inconsistency!


### Takeaway from Notebook 1
1. Evaluator scores are always lower than or equal to true inconsistency ($J^\alpha \le I$).
2. An All-Green report guarantees **only** that no defects exist *outside* the evaluators' blind spots.
3. In the next notebook, we construct a fully watched 6-site system where **every interface is monitored**, yet the system is All-Green while true inconsistency is arbitrarily large!